
Warehouse Delivery Robot - Search Algorithms (UCS & A*)
=============================================================================

Instructions on how to run the code:
1. Ensure you are in a Jupyter Notebook environment (e.g., Google Colab).
2. No external path-finding libraries are required. We only use built-in
   Python modules ('heapq' and 'time').
3. Run all cells sequentially from top to bottom (Runtime -> Run all).
4. The final cell will automatically execute both UCS and A* on all three
   maps, print the paths, display the maps, and generate a comparison table.

How to change the map variable:
- Below are three predefined maps (MAP_A, MAP_B, MAP_C).
- By default, the active map is assigned to the variable 'Grid' at the bottom
  of this cell.
- To test a different map for the individual display functions, simply change
  the assignment.
  For example, change `Grid = MAP_A` to `Grid = MAP_B`.
=============================================================================


In [ ]:
MAP_A = [
    "S..#....",
    "##.#.##.",
    "....~~..",
    ".####.#.",
    "...~..#G"
]

MAP_B = [
    "S~~~~~~~G",
    ".#######.",
    ".........",
    ".#.#.#.#.",
    "........."
]

MAP_C = [
    "S..#....",
    ".#.#.##.",
    "...#..#.",
    ".###.##.",
    "....#..G"
]

Grid = MAP_A

In [ ]:
# find start and goal
def find_positions(grid):
    start = None
    goal = None

    for r in range(len(grid)):
        for c in range(len(grid[r])):

            if grid[r][c] == "S":
                start = (r, c)

            elif grid[r][c] == "G":
                goal = (r, c)

    if start is None or goal is None:
        raise ValueError("The map must contain both S and G.")

    return start, goal

In [ ]:
# check cell validity
def is_valid_cell(grid, r, c):
    if r < 0 or r >= len(grid):
        return False
    if c < 0 or c >= len(grid[r]):
        return False
    if grid[r][c] == "#":
        return False

    return True

In [ ]:
# cost calculation
def move_cost(grid, r, c):
    if grid[r][c] == "~":
        return 3

    return 1

In [ ]:
def get_neighbors(grid, position):

    r, c = position

    directions = [
        (-1, 0), # up
        (1, 0), # down
        (0, -1), # left
        (0, 1) # right
    ]

    neighbors = []

    for dr, dc in directions:

        # Calculate the new position
        nr = r + dr
        nc = c + dc

        # Add the neighbor only if it is valid
        if is_valid_cell(grid, nr, nc):

            cost = move_cost(grid, nr, nc)

            neighbors.append(((nr, nc), cost))

    return neighbors

In [ ]:
# the whole path cost calculation
def calculate_path_cost(grid, path):
    total_cost = 0

    # Skip the starting position
    for r, c in path[1:]:
        total_cost += move_cost(grid, r, c)

    return total_cost

In [ ]:
# map display
def display_map(grid, path=None):

    displayed_grid = [list(row) for row in grid]

    if path is not None:

        for r, c in path:

            if displayed_grid[r][c] not in ("S", "G"):
                displayed_grid[r][c] = "*"

    for row in displayed_grid:
        print("".join(row))

In [ ]:
if __name__ == "__main__":
    start, goal = find_positions(Grid)

    print("Selected Map:")
    display_map(Grid)

    print("Start Position:", start)
    print("Goal Position:", goal)

    print("Valid Neighbors of the Start:")

    for position, cost in get_neighbors(Grid, start):
        print("Position:", position, ", Entry Cost:", cost)

Selected Map:
S..#....
##.#.##.
....~~..
.####.#.
...~..#G
Start Position: (0, 0)
Goal Position: (4, 7)
Valid Neighbors of the Start:
Position: (0, 1) , Entry Cost: 1


In [ ]:
import heapq   # priority queue (only a data structure, NOT a path-finding library)
import time

def reconstruct_path(parent, goal):
    """Walk back from the goal to the start using the parent dictionary."""
    path = []
    node = goal
    while node is not None:
        path.append(node)
        node = parent[node]
    path.reverse()
    return path


def ucs(grid):
    """
    Uniform Cost Search.
    Returns a dictionary with: path, cost, nodes_expanded, max_frontier, time.
    path is None if there is no solution.
    """
    start, goal = find_positions(grid)

    counter = 0                          # tie-breaker so heap never compares tuples of positions
    frontier = [(0, counter, start)]     # entries: (g cost, counter, position)
    parent = {start: None}               # to rebuild the path
    best_cost = {start: 0}               # cheapest known cost to reach each cell
    explored = set()

    nodes_expanded = 0
    max_frontier = 1

    t0 = time.perf_counter()

    while frontier:
        cost, _, current = heapq.heappop(frontier)

        # Skip outdated entries (a cheaper path to this cell was already found)
        if current in explored or cost > best_cost[current]:
            continue

        explored.add(current)
        nodes_expanded += 1              # the goal node is also counted when it is popped

        # Goal test when the node is POPPED (this is what guarantees optimality)
        if current == goal:
            elapsed = time.perf_counter() - t0
            return {"path": reconstruct_path(parent, goal), "cost": cost,
                    "nodes_expanded": nodes_expanded,
                    "max_frontier": max_frontier, "time": elapsed}

        # Expand the current node
        for neighbor, step_cost in get_neighbors(grid, current):
            new_cost = cost + step_cost
            if neighbor not in explored and (neighbor not in best_cost or new_cost < best_cost[neighbor]):
                best_cost[neighbor] = new_cost
                parent[neighbor] = current
                counter += 1
                heapq.heappush(frontier, (new_cost, counter, neighbor))

        max_frontier = max(max_frontier, len(frontier))

    # Frontier is empty and the goal was never reached
    elapsed = time.perf_counter() - t0
    return {"path": None, "cost": None, "nodes_expanded": nodes_expanded,
            "max_frontier": max_frontier, "time": elapsed}

In [ ]:
def run_ucs(grid, map_name=""):
    """Run UCS on a map, print all required outputs, and return the result dict."""
    print("=" * 50)
    print("Algorithm: Uniform Cost Search (UCS)", f"| Map: {map_name}" if map_name else "")
    print("=" * 50)

    result = ucs(grid)

    if result["path"] is None:
        print("No path found")
    else:
        print("Path:", result["path"])
        print("Path length (steps):", len(result["path"]) - 1)
        print("Total path cost:", result["cost"])
        # cross-check with Part 1's cost function
        assert result["cost"] == calculate_path_cost(grid, result["path"])
        print("Path on map:")
        display_map(grid, result["path"])

    print("Nodes expanded:", result["nodes_expanded"])
    print("Max frontier size:", result["max_frontier"])
    print(f"Run time: {result['time']*1000:.4f} ms")
    print()
    return result

In [ ]:
# Run UCS on the three maps
ucs_results = {}
for name, g in [("MAP_A", MAP_A), ("MAP_B", MAP_B), ("MAP_C", MAP_C)]:
    ucs_results[name] = run_ucs(g, name)

Algorithm: Uniform Cost Search (UCS) | Map: MAP_A
Path: [(0, 0), (0, 1), (0, 2), (1, 2), (2, 2), (2, 3), (2, 4), (2, 5), (2, 6), (2, 7), (3, 7), (4, 7)]
Path length (steps): 11
Total path cost: 15
Path on map:
S**#....
##*#.##.
..******
.####.#*
...~..#G
Nodes expanded: 28
Max frontier size: 4
Run time: 0.1064 ms

Algorithm: Uniform Cost Search (UCS) | Map: MAP_B
Path: [(0, 0), (1, 0), (2, 0), (2, 1), (2, 2), (2, 3), (2, 4), (2, 5), (2, 6), (2, 7), (2, 8), (1, 8), (0, 8)]
Path length (steps): 12
Total path cost: 12
Path on map:
S~~~~~~~G
*#######*
*********
.#.#.#.#.
.........
Nodes expanded: 31
Max frontier size: 4
Run time: 0.0894 ms

Algorithm: Uniform Cost Search (UCS) | Map: MAP_C
No path found
Nodes expanded: 13
Max frontier size: 3
Run time: 0.0337 ms



In [ ]:
# Results table for the report (UCS)
print(f"{'Map':<7}{'Path len':<10}{'Cost':<7}{'Expanded':<10}{'Max frontier':<14}{'Time (ms)':<10}")
for name, r in ucs_results.items():
    plen = len(r["path"]) - 1 if r["path"] else "-"
    cost = r["cost"] if r["path"] else "-"
    print(f"{name:<7}{str(plen):<10}{str(cost):<7}{r['nodes_expanded']:<10}{r['max_frontier']:<14}{r['time']*1000:<10.4f}")

Map    Path len  Cost   Expanded  Max frontier  Time (ms) 
MAP_A  11        15     28        4             0.1064    
MAP_B  12        12     31        4             0.0894    
MAP_C  -         -      13        3             0.0337    


In [ ]:
def manhattan_distance(pos, goal):
    """Calculate Manhattan distance between a position and the goal."""
    return abs(pos[0] - goal[0]) + abs(pos[1] - goal[1])

In [ ]:
def a_star(grid):
    """
    A* Search Algorithm using Manhattan Distance.
    Returns a dictionary with: path, cost, nodes_expanded, max_frontier, time.
    """
    start, goal = find_positions(grid)

    counter = 0
    # frontier entries: (f_cost, counter, g_cost, position)
    g_start = 0
    f_start = g_start + manhattan_distance(start, goal)
    frontier = [(f_start, counter, g_start, start)]
    parent = {start: None}
    best_g_cost = {start: g_start}
    explored = set()

    nodes_expanded = 0
    max_frontier = 1

    t0 = time.perf_counter()

    while frontier:
        f_cost, _, g_cost, current = heapq.heappop(frontier)

        # Skip outdated entries
        if current in explored or g_cost > best_g_cost[current]:
            continue

        explored.add(current)
        nodes_expanded += 1

        if current == goal:
            elapsed = time.perf_counter() - t0
            return {"path": reconstruct_path(parent, goal), "cost": g_cost,
                    "nodes_expanded": nodes_expanded,
                    "max_frontier": max_frontier, "time": elapsed}

        for neighbor, step_cost in get_neighbors(grid, current):
            new_g_cost = g_cost + step_cost
            if neighbor not in explored and (neighbor not in best_g_cost or new_g_cost < best_g_cost[neighbor]):
                best_g_cost[neighbor] = new_g_cost
                parent[neighbor] = current

                # Calculate f(n) = g(n) + h(n)
                f_cost_neighbor = new_g_cost + manhattan_distance(neighbor, goal)
                counter += 1
                heapq.heappush(frontier, (f_cost_neighbor, counter, new_g_cost, neighbor))

        max_frontier = max(max_frontier, len(frontier))

    elapsed = time.perf_counter() - t0
    return {"path": None, "cost": None, "nodes_expanded": nodes_expanded,
            "max_frontier": max_frontier, "time": elapsed}

In [ ]:
def run_a_star(grid, map_name=""):
    print("=" * 50)
    print("Algorithm: A* Search", f"| Map: {map_name}" if map_name else "")
    print("=" * 50)

    result = a_star(grid)

    if result["path"] is None:
        print("No path found")
    else:
        print("Path:", result["path"])
        print("Path length (steps):", len(result["path"]) - 1)
        print("Total path cost:", result["cost"])
        print("Path on map:")
        display_map(grid, result["path"])

    print("Nodes expanded:", result["nodes_expanded"])
    print("Max frontier size:", result["max_frontier"])
    print(f"Run time: {result['time']*1000:.4f} ms\n")
    return result

# Run A* and save results
astar_results = {}
for name, g in [("MAP_A", MAP_A), ("MAP_B", MAP_B), ("MAP_C", MAP_C)]:
    astar_results[name] = run_a_star(g, name)

# Final Comparison Table
print("="*85)
print(f"{'Map':<7} | {'Algorithm':<5} | {'Path len':<10} | {'Cost':<6} | {'Expanded':<9} | {'Max frontier':<13} | {'Time (ms)':<10}")
print("-" * 85)
for name in ["MAP_A", "MAP_B", "MAP_C"]:
    for algo, res_dict in [("UCS", ucs_results), ("A*", astar_results)]:
        r = res_dict[name]
        plen = len(r["path"]) - 1 if r["path"] else "-"
        cost = r["cost"] if r["path"] else "-"
        print(f"{name:<7} | {algo:<5} | {str(plen):<10} | {str(cost):<6} | {r['nodes_expanded']:<9} | {r['max_frontier']:<13} | {r['time']*1000:.4f}")
print("="*85)

Algorithm: A* Search | Map: MAP_A
Path: [(0, 0), (0, 1), (0, 2), (1, 2), (2, 2), (2, 3), (2, 4), (2, 5), (2, 6), (2, 7), (3, 7), (4, 7)]
Path length (steps): 11
Total path cost: 15
Path on map:
S**#....
##*#.##.
..******
.####.#*
...~..#G
Nodes expanded: 21
Max frontier size: 5
Run time: 0.1314 ms

Algorithm: A* Search | Map: MAP_B
Path: [(0, 0), (1, 0), (2, 0), (2, 1), (2, 2), (2, 3), (2, 4), (2, 5), (2, 6), (2, 7), (2, 8), (1, 8), (0, 8)]
Path length (steps): 12
Total path cost: 12
Path on map:
S~~~~~~~G
*#######*
*********
.#.#.#.#.
.........
Nodes expanded: 15
Max frontier size: 7
Run time: 0.0689 ms

Algorithm: A* Search | Map: MAP_C
No path found
Nodes expanded: 13
Max frontier size: 3
Run time: 0.0735 ms

Map     | Algorithm | Path len   | Cost   | Expanded  | Max frontier  | Time (ms) 
-------------------------------------------------------------------------------------
MAP_A   | UCS   | 11         | 15     | 28        | 4             | 0.1064
MAP_A   | A*    | 11         | 15 